## Installs

This is the self-contained training notebook for **Group 35 Solution C**.


In [1]:
!pip install -q transformers accelerate sentencepiece

## Imports


In [2]:
import csv
import json
import math
import os
import random
import re
import shutil
import string
import tempfile
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint
from typing import Any, Iterable, Optional
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

## Paths and Run Settings


In [ ]:
# ── Workspace paths ──
WORKSPACE = Path('/content/group35_solution_c_train')  # All training outputs go here
DATA_DIR  = Path('/content/data')  # Upload train.csv and dev.csv here

WORKSPACE.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Which models to train in this run.
# Options: 'raw20ep' (Model 1 only), 'symmetry', 'hardneg', 'long_specialist',
#          'both' (hardneg + specialist), 'selected_final' (all 4 models sequentially)
RUN_STAGE = 'selected_final'

# If True, package each trained checkpoint into an inference bundle zip after training
AUTO_PACKAGE_BUNDLES = True

# Length bucket boundaries (total word count) used for pair feature computation
# and for filtering training data for the long/xlong specialist
DEFAULT_LENGTH_BUCKETS = (
    ('short',  120),
    ('medium', 200),
    ('long',   300),
    ('xlong',  None),
)


## Model Config

In [ ]:
# ── Model training configurations ──
# Each key maps to a full config dict consumed by the train() function.
# Models are trained sequentially; continuation models reference an init_checkpoint_path
# pointing to the best.pt of their parent model.

MODEL_CONFIGS = {
    'transfomer_roberta_20epochs': {
        'pretrained_model_name': 'roberta-base',
        'train_path': 'train.csv',
        'dev_path': 'dev.csv',
        'output_dir': 'category_c/outputs/transfomer_roberta_20epochs',
        'seed': 13,
        'max_length': 256,
        'batch_size': 8,
        'epochs': 20,
        'learning_rate': 2e-05,
        'weight_decay': 0.01,
        'warmup_ratio': 0.1,
        'gradient_accumulation_steps': 4,
        'max_grad_norm': 1.0,
        'early_stopping_patience': 3,
        'selection_metric': 'accuracy',
        'fp16': True,
        'num_workers': 0,
        'train_swap_augmentation': True,
        'symmetric_inference': True,
        'use_fast_tokenizer': True,
        'tokenizer_name': 'whitespace',
        'lowercase': False,
        'normalize_urls': False,
        'normalize_emails': False,
        'normalize_numbers': False,
    },
    'transfomer_roberta_symmetry': {
        'pretrained_model_name': 'roberta-base',
        'init_checkpoint_path': 'category_c/outputs/transfomer_roberta_20epochs/best.pt',
        'train_path': 'train.csv',
        'dev_path': 'dev.csv',
        'output_dir': 'category_c/outputs/transfomer_roberta_symmetry',
        'seed': 13,
        'max_length': 256,
        'batch_size': 8,
        'epochs': 3,
        'learning_rate': 6e-06,
        'weight_decay': 0.01,
        'warmup_ratio': 0.05,
        'gradient_accumulation_steps': 4,
        'max_grad_norm': 1.0,
        'early_stopping_patience': 2,
        'selection_metric': 'f1',
        'fp16': True,
        'num_workers': 0,
        'train_swap_augmentation': True,
        'symmetric_inference': True,
        'use_fast_tokenizer': True,
        'tokenizer_name': 'whitespace',
        'lowercase': False,
        'normalize_urls': False,
        'normalize_emails': False,
        'normalize_numbers': False,
        'hard_negative_mining': False,
        'hard_negative_overlap_threshold': 0.35,
        'hard_negative_weight': 2.5,
        'hard_negative_placeholder_weight': 2.0,
        'focal_gamma': 0.0,
        'symmetric_consistency_lambda': 0.1,
    },
    'transfomer_roberta_hard_negative_precision': {
        'pretrained_model_name': 'roberta-base',
        'init_checkpoint_path': 'category_c/outputs/transfomer_roberta_20epochs/best.pt',
        'train_path': 'train.csv',
        'dev_path': 'dev.csv',
        'output_dir': 'category_c/outputs/transfomer_roberta_hard_negative_precision',
        'seed': 13,
        'max_length': 256,
        'batch_size': 8,
        'epochs': 3,
        'learning_rate': 8e-06,
        'weight_decay': 0.01,
        'warmup_ratio': 0.05,
        'gradient_accumulation_steps': 4,
        'max_grad_norm': 1.0,
        'early_stopping_patience': 2,
        'selection_metric': 'f1',
        'fp16': True,
        'num_workers': 0,
        'train_swap_augmentation': True,
        'symmetric_inference': True,
        'use_fast_tokenizer': True,
        'tokenizer_name': 'whitespace',
        'lowercase': False,
        'normalize_urls': False,
        'normalize_emails': False,
        'normalize_numbers': False,
        'hard_negative_mining': True,
        'hard_negative_overlap_threshold': 0.35,
        'hard_negative_weight': 2.5,
        'hard_negative_placeholder_weight': 2.0,
        'focal_gamma': 1.5,
        'symmetric_consistency_lambda': 0.0,
    },
    'transformer_roberta_long_text': {
        'pretrained_model_name': 'roberta-base',
        'init_checkpoint_path': 'category_c/outputs/transfomer_roberta_hard_negative_precision/best.pt',
        'train_path': 'train.csv',
        'dev_path': 'dev.csv',
        'output_dir': 'category_c/outputs/transformer_roberta_long_text',
        'seed': 23,
        'max_length': 384,
        'batch_size': 4,
        'epochs': 3,
        'learning_rate': 6e-06,
        'weight_decay': 0.01,
        'warmup_ratio': 0.05,
        'gradient_accumulation_steps': 8,
        'max_grad_norm': 1.0,
        'early_stopping_patience': 2,
        'selection_metric': 'f1',
        'fp16': True,
        'num_workers': 0,
        'train_swap_augmentation': True,
        'symmetric_inference': True,
        'use_fast_tokenizer': True,
        'tokenizer_name': 'whitespace',
        'lowercase': False,
        'normalize_urls': False,
        'normalize_emails': False,
        'normalize_numbers': False,
        'hard_negative_mining': True,
        'hard_negative_overlap_threshold': 0.35,
        'hard_negative_weight': 2.5,
        'hard_negative_placeholder_weight': 2.0,
        'focal_gamma': 1.5,
        'symmetric_consistency_lambda': 0.0,
        'allowed_length_buckets': ['long', 'xlong'],
    },
}

## Constants

In [ ]:
# ── Compiled regex patterns for text normalisation and pair feature extraction ──

URL_RE = re.compile(r'(https?://\S+|www\.\S+|urllink)', flags=re.IGNORECASE)
EMAIL_RE= re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
NUMBER_RE = re.compile(r'\b\d+(?:[.,:/-]\d+)*\b')
REPEATED_CHAR_RE = re.compile(r'(.)\1{2,}')
REPEATED_PUNCT_RE = re.compile(r'([!?.,;:])\1+')




## Text Processing


In [ ]:
@dataclass
class AVExample:
    """A single authorship verification example with tokenised and preprocessed text."""
    text_1: list             # Whitespace-tokenised words for text 1
    text_2: list             # Whitespace-tokenised words for text 2
    processed_text_1: str    # Preprocessed full string for text 1
    processed_text_2: str    # Preprocessed full string for text 2
    label: Optional[int]     # 1 = same author, 0 = different author, None = unlabelled

def normalize_label(raw_label: Optional[str]) -> Optional[int]:
    """Convert a raw CSV label string ('0', '1', '0.0', '1.0') to int, or None if missing."""

    if raw_label is None or raw_label == '':
        return None
    normalized = raw_label.strip()
    if normalized in {'0', '0.0'}:
        return 0
    if normalized in {'1', '1.0'}:
        return 1
    try:
        return int(float(normalized))
    except ValueError as exc:
        raise ValueError(f'Unsupported label value: {raw_label}') from exc


def preprocess_text(

    text: str,
    lowercase: bool = True,
    normalize_urls: bool = False,
    normalize_emails: bool = False,
    normalize_numbers: bool = False,
) -> str:
    """Apply optional text normalisation: lowercasing, URL/email/number replacement."""
    processed = str(text)
    if lowercase:
        processed = processed.lower()
    if normalize_urls:
        processed = re.sub(r'(https?://\S+|www\.\S+|urllink)', ' <url> ', processed, flags=re.IGNORECASE)
    if normalize_emails:
        processed = re.sub(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b', ' <email> ', processed)
    if normalize_numbers:
        processed = re.sub(r'\b\d+(?:[.,:/-]\d+)*\b', ' <num> ', processed)
    return processed


def tokenize(
    text: str,
    tokenizer_name: str = 'whitespace',
    lowercase: bool = True,
    normalize_urls: bool = False,
    normalize_emails: bool = False,
    normalize_numbers: bool = False,
) -> list:
    """Tokenise preprocessed text into a word list using the specified strategy."""

    processed = preprocess_text(
        text,
        lowercase=lowercase,
        normalize_urls=normalize_urls,
        normalize_emails=normalize_emails,
        normalize_numbers=normalize_numbers,
    )
    if tokenizer_name == 'regex':
        tokens = re.findall(r'<url>|<email>|<num>|\w+|[^\w\s]', processed)
    elif tokenizer_name == 'whitespace':
        tokens = processed.split()
    else:
        raise ValueError(f'Unsupported tokenizer_name: {tokenizer_name}')
    return tokens if tokens else ['<empty>']


def load_examples(
    path,
    max_samples: Optional[int] = None,
    expect_labels: bool = True,
    tokenizer_name: str = 'whitespace',
    lowercase: bool = True,
    normalize_urls: bool = False,
    normalize_emails: bool = False,
    normalize_numbers: bool = False,
) -> list:
    """Load authorship verification pairs from a CSV file."""

    examples = []
    with Path(path).open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            label = normalize_label(row.get('label')) if expect_labels else None
            processed_text_1 = preprocess_text(
                row['text_1'],
                lowercase=lowercase,
                normalize_urls=normalize_urls,
                normalize_emails=normalize_emails,
                normalize_numbers=normalize_numbers,
            )
            processed_text_2 = preprocess_text(
                row['text_2'],
                lowercase=lowercase,
                normalize_urls=normalize_urls,
                normalize_emails=normalize_emails,
                normalize_numbers=normalize_numbers,
            )
            examples.append(AVExample(
                text_1=tokenize(
                    row['text_1'],
                    tokenizer_name=tokenizer_name,
                    lowercase=lowercase,
                    normalize_urls=normalize_urls,
                    normalize_emails=normalize_emails,
                    normalize_numbers=normalize_numbers,
                ),
                text_2=tokenize(
                    row['text_2'],
                    tokenizer_name=tokenizer_name,
                    lowercase=lowercase,
                    normalize_urls=normalize_urls,
                    normalize_emails=normalize_emails,
                    normalize_numbers=normalize_numbers,
                ),
                processed_text_1=processed_text_1,
                processed_text_2=processed_text_2,
                label=label,
            ))
            if max_samples is not None and len(examples) >= max_samples:
                break
    return examples


def pad_or_truncate(token_ids: list, max_len: int, pad_id: int):
    """Truncate or pad a token ID list to exactly max_len. Returns (ids, attention_mask)."""

    if not token_ids:
        token_ids = [pad_id]
    token_ids = token_ids[:max_len]
    attention_mask = [1] * len(token_ids)
    if len(token_ids) < max_len:
        pad_count = max_len - len(token_ids)
        token_ids = token_ids + [pad_id] * pad_count
        attention_mask = attention_mask + [0] * pad_count
    return token_ids, attention_mask

## Pair Features


In [ ]:

def safe_div(numerator: float, denominator: float) -> float:
    if denominator == 0:
        return 0.0
    return numerator / denominator


def compute_length_bucket(total_words: int) -> str:
    """Map total word count of a pair to a length bucket name ('short'/'medium'/'long'/'xlong')."""

    for bucket_name, max_total_words in DEFAULT_LENGTH_BUCKETS:
        if max_total_words is None or total_words <= max_total_words:
            return bucket_name
    return 'xlong'


def token_jaccard(tokens_1: Iterable, tokens_2: Iterable) -> float:
    """Compute Jaccard similarity between two token sets: |intersection| / |union|."""

    set_1 = set(tokens_1)
    set_2 = set(tokens_2)
    union = set_1 | set_2
    if not union:
        return 0.0
    return len(set_1 & set_2) / len(union)


def char_ngram_jaccard(text_1: str, text_2: str, n: int) -> float:
    """Compute Jaccard similarity over character n-gram sets of two texts."""

    if n <= 0:
        raise ValueError('n must be positive.')

    def build_ngrams(text: str) -> set:
        compact = str(text).strip()
        if len(compact) < n:
            return {compact} if compact else set()
        return {compact[i: i + n] for i in range(len(compact) - n + 1)}

    grams_1 = build_ngrams(text_1)
    grams_2 = build_ngrams(text_2)
    union = grams_1 | grams_2
    if not union:
        return 0.0
    return len(grams_1 & grams_2) / len(union)


def punctuation_ratio(text: str) -> float:
    text = str(text)
    return safe_div(sum(1 for ch in text if ch in string.punctuation), len(text))


def uppercase_ratio(text: str) -> float:
    text = str(text)
    letters = [ch for ch in text if ch.isalpha()]
    return safe_div(sum(1 for ch in letters if ch.isupper()), len(letters))


def digit_ratio(text: str) -> float:
    text = str(text)
    return safe_div(sum(1 for ch in text if ch.isdigit()), len(text))


def average_word_length(text: str) -> float:
    words = [w for w in str(text).split() if w]
    if not words:
        return 0.0
    return sum(len(w) for w in words) / len(words)


def repeated_char_ratio(text: str) -> float:
    text = str(text)
    repeated_span_total = sum(len(m.group(0)) for m in REPEATED_CHAR_RE.finditer(text))
    return safe_div(repeated_span_total, len(text))


def repeated_punct_ratio(text: str) -> float:
    text = str(text)
    repeated_span_total = sum(len(m.group(0)) for m in REPEATED_PUNCT_RE.finditer(text))
    return safe_div(repeated_span_total, len(text))


def placeholder_counter(processed_text: str) -> Counter:
    tokens = str(processed_text).split()
    tracked = {'<url>', '<email>', '<num>'}
    return Counter(tok for tok in tokens if tok in tracked)


def placeholder_match_count(processed_text_1: str, processed_text_2: str) -> int:
    counter_1 = placeholder_counter(processed_text_1)
    counter_2 = placeholder_counter(processed_text_2)
    return sum(min(counter_1[tok], counter_2[tok]) for tok in {'<url>', '<email>', '<num>'})


def build_pair_feature_dict(text_1, text_2, processed_text_1, processed_text_2, tokens_1, tokens_2) -> dict:
    """Extract a dictionary of pair-level features for bucketing and hard-negative identification."""

    len_1 = len(tokens_1)
    len_2 = len(tokens_2)
    total_words = len_1 + len_2
    bucket = compute_length_bucket(total_words)

    avg_word_len_diff     = abs(average_word_length(text_1) - average_word_length(text_2))
    punct_ratio_diff      = abs(punctuation_ratio(text_1) - punctuation_ratio(text_2))
    uppercase_ratio_diff  = abs(uppercase_ratio(text_1) - uppercase_ratio(text_2))
    digit_ratio_diff      = abs(digit_ratio(text_1) - digit_ratio(text_2))
    rep_char_ratio_diff   = abs(repeated_char_ratio(text_1) - repeated_char_ratio(text_2))
    rep_punct_ratio_diff  = abs(repeated_punct_ratio(text_1) - repeated_punct_ratio(text_2))

    has_url_1    = int(bool(URL_RE.search(str(text_1))))
    has_url_2    = int(bool(URL_RE.search(str(text_2))))
    has_email_1  = int(bool(EMAIL_RE.search(str(text_1))))
    has_email_2  = int(bool(EMAIL_RE.search(str(text_2))))
    has_number_1 = int(bool(NUMBER_RE.search(str(text_1))))
    has_number_2 = int(bool(NUMBER_RE.search(str(text_2))))

    return {
        'len_1':                     float(len_1),
        'len_2':                     float(len_2),
        'total_words':               float(total_words),
        'abs_len_diff':              float(abs(len_1 - len_2)),
        'len_ratio':                 safe_div(min(len_1, len_2), max(len_1, len_2)),
        'token_jaccard':             token_jaccard(tokens_1, tokens_2),
        'char3_jaccard':             char_ngram_jaccard(text_1, text_2, 3),
        'char5_jaccard':             char_ngram_jaccard(text_1, text_2, 5),
        'avg_word_len_diff':         avg_word_len_diff,
        'punct_ratio_diff':          punct_ratio_diff,
        'uppercase_ratio_diff':      uppercase_ratio_diff,
        'digit_ratio_diff':          digit_ratio_diff,
        'repeated_char_ratio_diff':  rep_char_ratio_diff,
        'repeated_punct_ratio_diff': rep_punct_ratio_diff,
        'has_url_pair':              float(has_url_1 and has_url_2),
        'has_email_pair':            float(has_email_1 and has_email_2),
        'has_number_pair':           float(has_number_1 and has_number_2),
        'has_url_any':               float(has_url_1 or has_url_2),
        'has_email_any':             float(has_email_1 or has_email_2),
        'has_number_any':            float(has_number_1 or has_number_2),
        'placeholder_match_count':   float(placeholder_match_count(processed_text_1, processed_text_2)),
        'bucket_short':              float(bucket == 'short'),
        'bucket_medium':             float(bucket == 'medium'),
        'bucket_long':               float(bucket == 'long'),
        'bucket_xlong':              float(bucket == 'xlong'),
    }


def build_hard_negative_weight(label: Optional[int], feature_dict: dict, config: dict) -> float:
    """Compute a sample weight for hard-negative mining."""
    
    if label != 0 or not config.get('hard_negative_mining', False):
        return 1.0

    overlap_threshold        = float(config.get('hard_negative_overlap_threshold', 0.45))
    hard_negative_weight     = float(config.get('hard_negative_weight', 1.0))
    placeholder_bonus_weight = float(config.get('hard_negative_placeholder_weight', hard_negative_weight))

    special_pair_active = (
        feature_dict.get('has_url_pair', 0.0) > 0
        or feature_dict.get('has_email_pair', 0.0) > 0
        or feature_dict.get('has_number_pair', 0.0) > 0
        or feature_dict.get('placeholder_match_count', 0.0) > 0
    )

    weight = 1.0
    if feature_dict.get('token_jaccard', 0.0) >= overlap_threshold:
        weight = max(weight, hard_negative_weight)
    if special_pair_active:
        weight = max(weight, placeholder_bonus_weight)
    return weight

## Transformer Dataset


In [ ]:
def get_transformer_data_options(config: dict) -> dict:
    return {
        'tokenizer_name':    config.get('tokenizer_name', 'whitespace'),
        'lowercase':         config.get('lowercase', False),
        'normalize_urls':    config.get('normalize_urls', False),
        'normalize_emails':  config.get('normalize_emails', False),
        'normalize_numbers': config.get('normalize_numbers', False),
    }


def load_transformer_examples(path, config: dict, max_samples: Optional[int] = None, expect_labels: bool = True) -> list:
    """Load AV examples and optionally filter by allowed length buckets."""

    examples = load_examples(path, max_samples=max_samples, expect_labels=expect_labels, **get_transformer_data_options(config))

    allowed_length_buckets = config.get('allowed_length_buckets')
    if not allowed_length_buckets:
        return examples

    allowed = {str(b) for b in allowed_length_buckets}
    filtered_examples = []
    for example in examples:
        total_words = len(example.text_1) + len(example.text_2)
        if compute_length_bucket(total_words) in allowed:
            filtered_examples.append(example)
    return filtered_examples


class TransformerAVDataset(Dataset):
    """PyTorch Dataset that tokenises text pairs for the RoBERTa cross-encoder.
    
    Handles:
    - Pair encoding via the HuggingFace tokenizer (text_1 [SEP] text_2)
    - Swap augmentation: if enabled, the dataset is doubled (original + reversed pairs)
    - Reverse pair encoding: for symmetric consistency training, encodes (text_2, text_1) too
    - Hard-negative sample weighting: attaches a sample_weight tensor per example
    """

    def __init__(self, examples, tokenizer, max_length, include_reverse_pair=False, augment_swap=False, config=None):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.include_reverse_pair = include_reverse_pair
        self.augment_swap = augment_swap
        self.config = config or {}

    def __len__(self) -> int:
        if self.augment_swap:
            return len(self.examples) * 2
        return len(self.examples)

    def _resolve_pair(self, index: int):
        """Resolve the (example, text_1, text_2) for a given index, handling swap augmentation."""

        reverse_order = False
        base_index = index
        if self.augment_swap:
            reverse_order = index >= len(self.examples)
            base_index = index % len(self.examples)
        example = self.examples[base_index]
        text_1 = example.processed_text_1
        text_2 = example.processed_text_2
        if reverse_order:
            text_1, text_2 = text_2, text_1
        return example, text_1, text_2, reverse_order

    def _encode_pair(self, text_1: str, text_2: str) -> dict:
        """Tokenise a text pair into input_ids, attention_mask, etc. with truncation and padding."""

        encoded = self.tokenizer(text_1, text_2, truncation=True, max_length=self.max_length, padding='max_length')
        return {key: torch.tensor(value, dtype=torch.long) for key, value in encoded.items()}

    def __getitem__(self, index: int) -> dict:
        example, text_1, text_2, reverse_order = self._resolve_pair(index)
        item = self._encode_pair(text_1, text_2)

        pair_features = build_pair_feature_dict(
            text_1=example.processed_text_1,
            text_2=example.processed_text_2,
            processed_text_1=example.processed_text_1,
            processed_text_2=example.processed_text_2,
            tokens_1=example.text_1,
            tokens_2=example.text_2,
        )

        if self.include_reverse_pair:
            reverse_item = self._encode_pair(text_2, text_1)
            for key, value in reverse_item.items():
                item[f'reverse_{key}'] = value

        if example.label is not None:
            item['labels'] = torch.tensor(example.label, dtype=torch.long)
            item['sample_weight'] = torch.tensor(
                build_hard_negative_weight(example.label, pair_features, self.config),
                dtype=torch.float32,
            )

        return item

## Model and Checkpoint Utilities


In [ ]:
def resolve_hf_model_dir(path) -> Path:
    """Locate the HuggingFace model directory (containing config.json) near a checkpoint path."""

    path = Path(path)
    if path.is_dir() and (path / 'config.json').exists():
        return path
    if path.is_dir() and (path / 'hf_model').exists():
        return path / 'hf_model'
    candidate = path.parent / 'hf_model'
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Could not find a saved HuggingFace model directory for: {path}')


def build_transformer_model_and_tokenizer(config: dict):
    """Initialise a fresh roberta-base model and tokenizer for 2-class sequence classification."""

    model_name = config['pretrained_model_name']
    hf_config = AutoConfig.from_pretrained(model_name, num_labels=2, finetuning_task='authorship_verification')
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=config.get('use_fast_tokenizer', True))
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=hf_config)
    return model, tokenizer


def save_transformer_checkpoint(output_dir: Path, model, tokenizer, checkpoint_data: dict) -> Path:
    """Save model weights (best.pt) and HuggingFace model/tokenizer files to output_dir."""

    checkpoint_path = output_dir / 'best.pt'
    torch.save(checkpoint_data, checkpoint_path)
    hf_model_dir = output_dir / 'hf_model'
    model.save_pretrained(str(hf_model_dir))
    tokenizer.save_pretrained(str(hf_model_dir))
    return checkpoint_path


def load_transformer_bundle(checkpoint_path, device=None) -> dict:
    """Load a saved checkpoint + HuggingFace model into a ready-to-use inference bundle dict."""

    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    config = checkpoint['config']
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    hf_model_dir = resolve_hf_model_dir(checkpoint_path)
    tokenizer = AutoTokenizer.from_pretrained(str(hf_model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(hf_model_dir)).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    return {
        'checkpoint_path': checkpoint_path,
        'checkpoint': checkpoint,
        'config': config,
        'hf_model_dir': hf_model_dir,
        'tokenizer': tokenizer,
        'model': model,
        'device': device,
        'default_threshold': float(checkpoint.get('threshold', 0.5)),
    }


def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {key: value.to(device) for key, value in batch.items()}


def get_model_inputs(batch: dict, reverse: bool = False, view: str = 'main') -> dict:
    """Extract the correct input tensors from a batch for the model forward pass.
    
    Args:
        reverse: If True, select reverse-pair tensors (prefixed 'reverse_').
        view: 'main' for the primary encoding, 'alt' for dual-view encoding.
    Returns:
        Dict with keys like 'input_ids', 'attention_mask' (prefix stripped).
    """

    if view not in {'main', 'alt'}:
        raise ValueError(f'Unsupported view: {view}')

    if view == 'main':
        prefix = 'reverse_' if reverse else ''
    else:
        prefix = 'alt_reverse_' if reverse else 'alt_'

    ignored_keys = {'labels', 'sample_weight'}
    inputs = {}

    for key, value in batch.items():
        if key in ignored_keys:
            continue
        if prefix == '':
            if not key.startswith('reverse_') and not key.startswith('alt_'):
                inputs[key] = value
        elif key.startswith(prefix):
            inputs[key[len(prefix):]] = value

    return inputs


def probabilities_from_logits(logits: torch.Tensor) -> torch.Tensor:
    """Convert 2-class logits to P(same_author) by taking softmax[:, 1]."""

    return torch.softmax(logits, dim=-1)[:, 1]

## Helper functions

In [10]:
def ensure_dir(path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(data, path) -> None:
    with Path(path).open('w', encoding='utf-8') as handle:
        json.dump(data, handle, ensure_ascii=True, indent=2)


def load_json(path) -> dict:
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_classification_metrics(probabilities: list, labels: list, threshold: float) -> dict:
    predictions = [1 if prob >= threshold else 0 for prob in probabilities]
    tp = sum(1 for pred, label in zip(predictions, labels) if pred == 1 and label == 1)
    tn = sum(1 for pred, label in zip(predictions, labels) if pred == 0 and label == 0)
    fp = sum(1 for pred, label in zip(predictions, labels) if pred == 1 and label == 0)
    fn = sum(1 for pred, label in zip(predictions, labels) if pred == 0 and label == 1)
    total = max(len(labels), 1)
    accuracy  = (tp + tn) / total
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}


def find_best_threshold(probabilities: list, labels: list, metric_name: str = 'accuracy'):
    best_threshold = 0.5
    best_metrics   = compute_classification_metrics(probabilities, labels, threshold=0.5)
    best_score     = best_metrics[metric_name]
    for step in range(10, 91):
        threshold = step / 100.0
        metrics   = compute_classification_metrics(probabilities, labels, threshold=threshold)
        score     = metrics[metric_name]
        if score > best_score:
            best_threshold = threshold
            best_metrics   = metrics
            best_score     = score
    return best_threshold, best_metrics


def save_prediction_rows(path, probabilities: list, labels: list, threshold: float) -> None:
    lines = ['index,label,probability,prediction']
    for index, (label, probability) in enumerate(zip(labels, probabilities)):
        prediction = 1 if probability >= threshold else 0
        lines.append(f'{index},{label},{probability:.6f},{prediction}')
    Path(path).write_text('\n'.join(lines) + '\n', encoding='utf-8')

## Training Loop

In [ ]:
def format_metrics(metrics: dict) -> str:
    return (
        f"acc={metrics['accuracy']:.4f} "
        f"prec={metrics['precision']:.4f} "
        f"rec={metrics['recall']:.4f} "
        f"f1={metrics['f1']:.4f}"
    )


def summarize_thresholds(probabilities: list, labels: list) -> dict:
    best_accuracy_threshold, best_accuracy_metrics = find_best_threshold(probabilities, labels, metric_name='accuracy')
    best_f1_threshold, best_f1_metrics             = find_best_threshold(probabilities, labels, metric_name='f1')
    return {
        'accuracy': {'threshold': best_accuracy_threshold, 'metrics': best_accuracy_metrics},
        'f1':       {'threshold': best_f1_threshold,       'metrics': best_f1_metrics},
    }


def run_epoch(model, dataloader, optimizer, scheduler, device, gradient_accumulation_steps, max_grad_norm, use_amp, focal_gamma, symmetric_consistency_lambda) -> float:
    """Run one training epoch with optional focal loss and symmetric consistency.
    
    Training loss components:
    1. Cross-entropy on (text_1, text_2) — the forward pass
    2. (If focal_gamma > 0) Focal loss weighting: (1 - p_target)^γ × CE
    3. (If symmetric_consistency_lambda > 0) Reverse pass on (text_2, text_1),
       averaged with forward loss, plus a consistency penalty on P_fwd vs P_rev
    4. (If hard-negative mining) Per-example sample weights from the dataset
    
    Uses gradient accumulation, mixed-precision (AMP), and gradient clipping.
    
    Returns:
        Average training loss over all examples.
    """

    model.train()
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    optimizer.zero_grad(set_to_none=True)
    warned_about_amp_clipping = False
    total_loss = 0.0
    total_examples = 0

    for step, batch in enumerate(dataloader, start=1):
        batch = move_batch_to_device(batch, device)

        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
            # ── Forward pass ──
            outputs = model(**get_model_inputs(batch, view='main'))
            forward_logits = outputs.logits
            forward_loss   = F.cross_entropy(forward_logits, batch['labels'], reduction='none')
            forward_probs  = torch.softmax(forward_logits, dim=-1)

            # ── Focal loss: down-weight easy examples ──
            if focal_gamma > 0:
                target_probs         = forward_probs.gather(1, batch['labels'].unsqueeze(1)).squeeze(1)
                focal_factor         = torch.pow(1.0 - target_probs, focal_gamma)
                forward_loss         = forward_loss * focal_factor

            per_example_loss    = forward_loss
            consistency_losses  = []

            # ── Symmetric consistency: also compute loss on reversed pair order ──
            if symmetric_consistency_lambda > 0:
                reverse_logits = model(**get_model_inputs(batch, reverse=True, view='main')).logits
                reverse_loss   = F.cross_entropy(reverse_logits, batch['labels'], reduction='none')
                reverse_probs  = torch.softmax(reverse_logits, dim=-1)

                if focal_gamma > 0:
                    reverse_target_probs = reverse_probs.gather(1, batch['labels'].unsqueeze(1)).squeeze(1)
                    reverse_focal_factor = torch.pow(1.0 - reverse_target_probs, focal_gamma)
                    reverse_loss         = reverse_loss * reverse_focal_factor

                # Average forward and reverse losses
                per_example_loss = 0.5 * (forward_loss + reverse_loss)
                # Penalise disagreement between forward and reverse predictions
                consistency_losses.append(
                    symmetric_consistency_lambda
                    * torch.square(forward_probs[:, 1] - reverse_probs[:, 1])
                )

            # ── Apply hard-negative sample weights ──
            if 'sample_weight' in batch:
                per_example_loss   = per_example_loss * batch['sample_weight']
                consistency_losses = [loss * batch['sample_weight'] for loss in consistency_losses]

            # ── Combine losses and scale for gradient accumulation ──
            raw_loss = per_example_loss.mean()
            if consistency_losses:
                raw_loss = raw_loss + sum(loss.mean() for loss in consistency_losses)
            loss = raw_loss / gradient_accumulation_steps

        # ── Backward pass + optimizer step every N accumulated steps ──
        scaler.scale(loss).backward()

        should_step = step % gradient_accumulation_steps == 0 or step == len(dataloader)
        if should_step:
            if max_grad_norm is not None and max_grad_norm > 0:
                try:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                except ValueError:
                    if not warned_about_amp_clipping:
                        print('Warning: skipping gradient clipping because AMP unscale is not supported.')
                        warned_about_amp_clipping = True
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()

        batch_size = batch['labels'].size(0)
        total_loss += raw_loss.item() * batch_size
        total_examples += batch_size

    return total_loss / max(total_examples, 1)


@torch.no_grad()
def evaluate(model, dataloader, device, symmetric_inference) -> tuple:
    """Evaluate model on a labelled dataset, returning (avg_loss, probabilities, labels).
    
    If symmetric_inference is True, averages predictions from both pair orderings.
    """

    model.eval()
    total_loss     = 0.0
    total_examples = 0
    probabilities  = []
    labels         = []

    for batch in dataloader:
        batch   = move_batch_to_device(batch, device)
        outputs = model(**get_model_inputs(batch, view='main'), labels=batch['labels'])
        probs   = probabilities_from_logits(outputs.logits)

        if symmetric_inference:
            reverse_outputs = model(**get_model_inputs(batch, reverse=True, view='main'))
            reverse_probs   = probabilities_from_logits(reverse_outputs.logits)
            probs           = 0.5 * (probs + reverse_probs)

        batch_size = batch['labels'].size(0)
        total_loss += outputs.loss.item() * batch_size
        total_examples += batch_size
        probabilities.extend(probs.cpu().tolist())
        labels.extend(batch['labels'].cpu().tolist())

    return total_loss / max(total_examples, 1), probabilities, labels


def train(config: dict) -> None:
    """Full training pipeline for a single transformer cross-encoder model.
    
    Steps:
    1. Load and preprocess training/dev data (with optional length-bucket filtering)
    2. Build model from pretrained backbone (or load from init_checkpoint_path)
    3. Train with AdamW + linear warmup/decay + early stopping
    4. Save best checkpoint (by selection_metric) and dev predictions
    
    The config dict controls all hyperparameters: see MODEL_CONFIGS above.
    """
    
    set_seed(config['seed'])
    output_dir = ensure_dir(config['output_dir'])
    save_json(config, output_dir / 'config_resolved.json')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    print('Loading data...')
    train_examples = load_transformer_examples(config['train_path'], config=config, expect_labels=True)
    dev_examples   = load_transformer_examples(config['dev_path'],   config=config, expect_labels=True)
    print(f'Train: {len(train_examples)}  Dev: {len(dev_examples)}')

    print('Loading tokenizer and model...')
    model, tokenizer = build_transformer_model_and_tokenizer(config)
    model = model.to(device)

    init_checkpoint_path = config.get('init_checkpoint_path')
    if init_checkpoint_path:
        init_checkpoint = torch.load(init_checkpoint_path, map_location='cpu')
        model.load_state_dict(init_checkpoint['model_state_dict'])
        print(f'Initialized weights from: {init_checkpoint_path}')

    symmetric_inference            = config.get('symmetric_inference', False)
    focal_gamma                    = float(config.get('focal_gamma', 0.0))
    symmetric_consistency_lambda   = float(config.get('symmetric_consistency_lambda', 0.0))

    train_dataset = TransformerAVDataset(
        train_examples,
        tokenizer=tokenizer,
        max_length=config['max_length'],
        include_reverse_pair=symmetric_consistency_lambda > 0,
        augment_swap=config.get('train_swap_augmentation', False),
        config=config,
    )
    dev_dataset = TransformerAVDataset(
        dev_examples,
        tokenizer=tokenizer,
        max_length=config['max_length'],
        include_reverse_pair=symmetric_inference,
        augment_swap=False,
        config=config,
    )

    loader_kwargs = {
        'batch_size':  config['batch_size'],
        'num_workers': config.get('num_workers', 0),
        'pin_memory':  device.type == 'cuda',
    }
    train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kwargs)
    dev_loader   = DataLoader(dev_dataset,   shuffle=False, **loader_kwargs)

    optimizer = AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config.get('weight_decay', 0.0))

    gradient_accumulation_steps = config.get('gradient_accumulation_steps', 1)
    total_update_steps = math.ceil(len(train_loader) / gradient_accumulation_steps) * config['epochs']
    warmup_steps       = int(total_update_steps * config.get('warmup_ratio', 0.0))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_update_steps)

    use_amp          = bool(config.get('fp16', False) and device.type == 'cuda')
    selection_metric = config.get('selection_metric', 'f1')
    patience         = config.get('early_stopping_patience', 2)
    max_grad_norm    = config.get('max_grad_norm', 1.0)

    best_score              = float('-inf')
    best_epoch              = 0
    best_threshold          = 0.5
    best_threshold_summary  = None
    best_probabilities      = []
    best_labels             = []
    epochs_without_improvement = 0
    history                 = []

    print('Training...')
    for epoch in range(1, config['epochs'] + 1):
        train_loss = run_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            gradient_accumulation_steps=gradient_accumulation_steps,
            max_grad_norm=max_grad_norm,
            use_amp=use_amp,
            focal_gamma=focal_gamma,
            symmetric_consistency_lambda=symmetric_consistency_lambda,
        )
        dev_loss, probabilities, labels = evaluate(
            model=model,
            dataloader=dev_loader,
            device=device,
            symmetric_inference=symmetric_inference,
        )

        default_metrics    = compute_classification_metrics(probabilities, labels, threshold=0.5)
        threshold_summary  = summarize_thresholds(probabilities, labels)
        accuracy_summary   = threshold_summary['accuracy']
        f1_summary         = threshold_summary['f1']
        selected_summary   = threshold_summary[selection_metric]
        selected_score     = selected_summary['metrics'][selection_metric]

        print(
            f"Epoch {epoch}/{config['epochs']} "
            f"train_loss={train_loss:.4f} dev_loss={dev_loss:.4f} "
            f"default[{format_metrics(default_metrics)}] "
            f"acc@{accuracy_summary['threshold']:.2f}[{format_metrics(accuracy_summary['metrics'])}] "
            f"f1@{f1_summary['threshold']:.2f}[{format_metrics(f1_summary['metrics'])}]"
        )

        history.append({
            'epoch': epoch, 'train_loss': train_loss, 'dev_loss': dev_loss,
            'default_metrics': default_metrics, 'threshold_summary': threshold_summary,
        })

        if selected_score > best_score:
            best_score             = selected_score
            best_epoch             = epoch
            best_threshold         = selected_summary['threshold']
            best_threshold_summary = threshold_summary
            best_probabilities     = list(probabilities)
            best_labels            = list(labels)
            epochs_without_improvement = 0
            checkpoint_data = {
                'model_state_dict':    model.state_dict(),
                'config':              config,
                'epoch':               epoch,
                'selection_metric':    selection_metric,
                'threshold':           best_threshold,
                'metrics':             selected_summary['metrics'],
                'threshold_summary':   threshold_summary,
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
            }
            save_transformer_checkpoint(output_dir, model, tokenizer, checkpoint_data)
            save_prediction_rows(output_dir / 'dev_predictions.csv', best_probabilities, best_labels, best_threshold)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f'Early stopping triggered after epoch {epoch}.')
            break

    save_json(history, output_dir / 'history.json')
    save_json({
        'best_epoch': best_epoch, 'selection_metric': selection_metric,
        'best_threshold': best_threshold, 'best_threshold_summary': best_threshold_summary,
    }, output_dir / 'summary.json')

    print(f'Done. Best epoch={best_epoch}, {selection_metric}={best_score:.4f}, threshold={best_threshold:.2f}')

## Training Execution



In [ ]:
# ── Copy data to workspace and start training ──

for name in ['train.csv', 'dev.csv']:
    shutil.copy2(DATA_DIR / name, WORKSPACE / name)

os.chdir(str(WORKSPACE))

# Maps each RUN_STAGE option to the list of models to train (in dependency order)
STAGE_MODEL_ORDER = {
    'raw20ep':         ['transfomer_roberta_20epochs'],
    'symmetry':        ['transfomer_roberta_symmetry'],
    'hardneg':         ['transfomer_roberta_hard_negative_precision'],
    'long_specialist': ['transformer_roberta_long_text'],
    'both': [
        'transfomer_roberta_hard_negative_precision',
        'transformer_roberta_long_text',
    ],
    'selected_final': [
        'transfomer_roberta_20epochs',
        'transfomer_roberta_symmetry',
        'transfomer_roberta_hard_negative_precision',
        'transformer_roberta_long_text',
    ],
}

# Train each model sequentially
for model_name in STAGE_MODEL_ORDER[RUN_STAGE]:
    print(f'\n{"="*60}\nTraining: {model_name}\n{"="*60}')
    train(MODEL_CONFIGS[model_name])


Training: transfomer_roberta_20epochs
Device: cuda
Loading data...
Train: 27643  Dev: 5993
Loading tokenizer and model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3/20 train_loss=0.3700 dev_loss=0.4921 default[acc=0.7934 prec=0.8244 rec=0.7559 f1=0.7887] acc@0.51[acc=0.7954 prec=0.8306 rec=0.7523 f1=0.7895] f1@0.19[acc=0.7856 prec=0.7420 rec=0.8884 f1=0.8086]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4/20 train_loss=0.2378 dev_loss=0.5586 default[acc=0.7939 prec=0.7832 rec=0.8240 f1=0.8031] acc@0.66[acc=0.7976 prec=0.8163 rec=0.7781 f1=0.7968] f1@0.32[acc=0.7878 prec=0.7511 rec=0.8730 f1=0.8075]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]